Feature Emgineering

In [0]:
# Import required libraries
from pyspark.sql.functions import to_date, months_between, col, current_date
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# Load Data
df = spark.table("data_processed.credit_risk_main.loans_transformed")

In [0]:
# Step 1: Ensure earliest_cr_line is in proper date format
# Assuming earliest_cr_line is like "01/2005" or "MM/yyyy"
df = df.withColumn(
    "earliest_cr_line_parsed",
    to_date(col("earliest_cr_line_parsed"), "MM/yyyy")
)

In [0]:
# Step 2: Calculate credit age in years
df = df.withColumn(
    "credit_age_years",
    months_between(current_date(), col("earliest_cr_line_parsed")) / 12
)

In [0]:
# Optional: Round to 2 decimal places
from pyspark.sql.functions import round
df = df.withColumn("credit_age_years", round(col("credit_age_years"), 2))



In [0]:
# Step 3: Show sample
df.select("earliest_cr_line_parsed", "credit_age_years").show(10)

+-----------------------+----------------+
|earliest_cr_line_parsed|credit_age_years|
+-----------------------+----------------+
|             2005-01-06|           20.73|
|             1994-01-08|           31.73|
|             1999-01-09|           26.72|
|             2002-01-08|           23.73|
|             2000-01-04|           25.74|
|             1990-01-09|           35.72|
|             1992-01-01|           33.74|
|             2007-01-08|           18.73|
|             1996-01-10|           29.72|
|             2006-01-12|           19.72|
+-----------------------+----------------+
only showing top 10 rows


In [0]:
# Income to Loan ratio

from pyspark.sql.functions import col, round

# Step 1: Ensure both columns exist
required_cols = ["annual_inc", "loan_amnt"]
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    print(f"Missing columns: {missing_cols}")
else:
    # Step 2: Calculate Income-to-Loan Ratio
    df = df.withColumn(
        "income_to_loan_ratio",
        round(col("annual_inc") / col("loan_amnt"), 2)
    )
    
    # Step 3: Show sample
    df.select("annual_inc", "loan_amnt", "income_to_loan_ratio").show(10)


+----------+---------+--------------------+
|annual_inc|loan_amnt|income_to_loan_ratio|
+----------+---------+--------------------+
|   80000.0|  16000.0|                 5.0|
|  110000.0|   5500.0|                20.0|
|   66000.0|  14000.0|                4.71|
|   80000.0|  14000.0|                5.71|
|   35000.0|  10000.0|                 3.5|
|   33000.0|  12000.0|                2.75|
|   75000.0|   5400.0|               13.89|
|   51360.0|  10000.0|                5.14|
|  160000.0|  22200.0|                7.21|
|   52800.0|   9000.0|                5.87|
+----------+---------+--------------------+
only showing top 10 rows


In [0]:
df.columns

['id',
 'member_id',
 'issue_date',
 'earliest_cr_line_parsed',
 'lastpymt_date_parsed',
 'lastcreditpull_date_parsed',
 'next_payment_date_parsed',
 'default_ind',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_length_num',
 'emp_title',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'pymnt_plan',
 'desc',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_amnt',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'policy_code',
 'application_type',
 'annual_inc_joint',
 'dti_joint',
 'verification_status_joint',
 'acc_now_de

In [0]:
# Categorizing Risk Ratio
from pyspark.sql.functions import when, col
# Step 1: Ensure the column exists
if "income_to_loan_ratio" in df.columns:
    # Step 2: Create risk category
    df = df.withColumn(
        "income_loan_risk",
        when(col("income_to_loan_ratio") < 1, "High Risk")
        .when((col("income_to_loan_ratio") >= 1) & (col("income_to_loan_ratio") <= 3), "Medium Risk")
        .otherwise("Low Risk")
    )
    
    # Step 3: Show sample
    df.select("annual_inc", "loan_amnt", "income_to_loan_ratio", "income_loan_risk").show(10)
else:
    print("Column 'income_to_loan_ratio' not found in DataFrame")

+----------+---------+--------------------+----------------+
|annual_inc|loan_amnt|income_to_loan_ratio|income_loan_risk|
+----------+---------+--------------------+----------------+
|   80000.0|  16000.0|                 5.0|        Low Risk|
|  110000.0|   5500.0|                20.0|        Low Risk|
|   66000.0|  14000.0|                4.71|        Low Risk|
|   80000.0|  14000.0|                5.71|        Low Risk|
|   35000.0|  10000.0|                 3.5|        Low Risk|
|   33000.0|  12000.0|                2.75|     Medium Risk|
|   75000.0|   5400.0|               13.89|        Low Risk|
|   51360.0|  10000.0|                5.14|        Low Risk|
|  160000.0|  22200.0|                7.21|        Low Risk|
|   52800.0|   9000.0|                5.87|        Low Risk|
+----------+---------+--------------------+----------------+
only showing top 10 rows


In [0]:
# Debt to income ratio
from pyspark.sql.functions import col, round

# Ensure required columns exist
required_cols = ["installment", "annual_inc"]
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    print(f"Missing columns: {missing_cols}")
else:
    # Calculate DTI ratio as a percentage
    df = df.withColumn(
        "debt_to_income_ratio",
        round((col("installment") * 12) / col("annual_inc") * 100, 2)
    )
    
    # Show sample
    df.select("installment", "annual_inc", "dti", "debt_to_income_ratio").show(10)



+-----------+----------+-----+--------------------+
|installment|annual_inc|  dti|debt_to_income_ratio|
+-----------+----------+-----+--------------------+
|     379.39|   80000.0|19.59|                5.69|
|     168.88|  110000.0| 9.69|                1.84|
|     429.86|   66000.0|11.15|                7.82|
|     480.33|   80000.0|16.21|                 7.2|
|     335.45|   35000.0|12.03|                11.5|
|      386.7|   33000.0| 18.8|               14.06|
|     181.15|   75000.0|  8.8|                 2.9|
|     353.01|   51360.0|16.61|                8.25|
|     818.49|  160000.0|13.33|                6.14|
|     281.62|   52800.0| 7.93|                 6.4|
+-----------+----------+-----+--------------------+
only showing top 10 rows


In [0]:
from pyspark.sql.functions import when, col

# Ensure the column exists
if "debt_to_income_ratio" in df.columns:
    # Create DTI risk category
    df = df.withColumn(
        "dti_risk",
        when(col("debt_to_income_ratio") < 10, "Low Risk")
        .when((col("debt_to_income_ratio") >= 10) & (col("debt_to_income_ratio") <= 25), "Medium Risk")
        .otherwise("High Risk")
    )
    
    # Show sample
    df.select("installment", "annual_inc", "debt_to_income_ratio", "dti_risk").show(10)
else:
    print("Column 'debt_to_income_ratio' not found in DataFrame")


+-----------+----------+--------------------+-----------+
|installment|annual_inc|debt_to_income_ratio|   dti_risk|
+-----------+----------+--------------------+-----------+
|     379.39|   80000.0|                5.69|   Low Risk|
|     168.88|  110000.0|                1.84|   Low Risk|
|     429.86|   66000.0|                7.82|   Low Risk|
|     480.33|   80000.0|                 7.2|   Low Risk|
|     335.45|   35000.0|                11.5|Medium Risk|
|      386.7|   33000.0|               14.06|Medium Risk|
|     181.15|   75000.0|                 2.9|   Low Risk|
|     353.01|   51360.0|                8.25|   Low Risk|
|     818.49|  160000.0|                6.14|   Low Risk|
|     281.62|   52800.0|                 6.4|   Low Risk|
+-----------+----------+--------------------+-----------+
only showing top 10 rows


In [0]:
# Distribution of borrowers across income-to-loan risk categories

from pyspark.sql.functions import col, round

# Ensure the column exists
if "income_loan_risk" in df.columns:
    # Count borrowers per risk category
    risk_distribution = (
        df.groupBy("income_loan_risk")
          .count()
          .withColumn("percentage", round(col("count") / df.count() * 100, 2))
    )
    
    # Show distribution
    risk_distribution.show()
else:
    print("Column 'income_loan_risk' not found in DataFrame")


+----------------+------+----------+
|income_loan_risk| count|percentage|
+----------------+------+----------+
|        Low Risk|694241|     81.11|
|     Medium Risk|161711|     18.89|
|       High Risk|    17|       0.0|
+----------------+------+----------+



In [0]:
# Disable ANSI mode
spark.conf.set("spark.sql.ansi.enabled", "false")


In [0]:
from pyspark.sql.functions import col, round

total_count = df.count()

dti_distribution = (
    df.groupBy("dti_risk")
      .count()
      .withColumn("percentage", round(col("count") / total_count * 100, 2))
)

dti_distribution.show()



+-----------+------+----------+
|   dti_risk| count|percentage|
+-----------+------+----------+
|   Low Risk|608230|     71.06|
|Medium Risk|247672|     28.93|
|  High Risk|    67|      0.01|
+-----------+------+----------+



In [0]:
# Calculating Revolving Utilization

from pyspark.sql.functions import col, round, expr

# Ensure required columns exist
required_cols = ["revol_bal", "total_rev_hi_lim"]
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    print(f"Missing columns: {missing_cols}")
else:
    # Option 1: Using try_divide to avoid division by zero
    df = df.withColumn(
        "revol_utilization",
        round(expr("try_divide(revol_bal, total_rev_hi_lim) * 100"), 2)  # percentage
    )
    
    # Optional: show sample
    df.select("revol_bal", "total_rev_hi_lim", "revol_utilization").show(10)


+---------+----------------+-----------------+
|revol_bal|total_rev_hi_lim|revol_utilization|
+---------+----------------+-----------------+
|  11113.0|         13300.0|            83.56|
|  14136.0|             0.0|             NULL|
|  12095.0|             0.0|             NULL|
|  11546.0|             0.0|             NULL|
|  10472.0|             0.0|             NULL|
|  17229.0|             0.0|             NULL|
|  10376.0|             0.0|             NULL|
|   8495.0|             0.0|             NULL|
|  10070.0|             0.0|             NULL|
|   3219.0|             0.0|             NULL|
+---------+----------------+-----------------+
only showing top 10 rows


In [0]:
# Replace nulls in revol_utilization with 0
df = df.fillna({"revol_utilization": 0})

# Show sample
df.select("revol_bal", "total_rev_hi_lim", "revol_utilization").show(10)


+---------+----------------+-----------------+
|revol_bal|total_rev_hi_lim|revol_utilization|
+---------+----------------+-----------------+
|  11113.0|         13300.0|            83.56|
|  14136.0|             0.0|              0.0|
|  12095.0|             0.0|              0.0|
|  11546.0|             0.0|              0.0|
|  10472.0|             0.0|              0.0|
|  17229.0|             0.0|              0.0|
|  10376.0|             0.0|              0.0|
|   8495.0|             0.0|              0.0|
|  10070.0|             0.0|              0.0|
|   3219.0|             0.0|              0.0|
+---------+----------------+-----------------+
only showing top 10 rows


In [0]:
# Late Payment History: Binary flag for borrowers with mths_since_last_delinq > 0.

from pyspark.sql.functions import when, col

# Ensure column exists
if "mths_since_last_delinq" in df.columns:
    # Replace nulls with 0 first if needed
    # df = df.fillna({"mths_since_last_delinq": 0})
    
    # Create binary flag: 1 if borrower had any delinquency, 0 otherwise
    df = df.withColumn(
        "late_payment_flag",
        when(col("mths_since_last_delinq") > 0, 1).otherwise(0)
    )
    
    # Show sample
    df.select("mths_since_last_delinq", "late_payment_flag").show(10)
else:
    print("Column 'mths_since_last_delinq' not found in DataFrame")


+----------------------+-----------------+
|mths_since_last_delinq|late_payment_flag|
+----------------------+-----------------+
|                     0|                0|
|                     0|                0|
|                     0|                0|
|                     0|                0|
|                     0|                0|
|                     0|                0|
|                    31|                1|
|                     0|                0|
|                    69|                1|
|                     0|                0|
+----------------------+-----------------+
only showing top 10 rows


Binary flag:

1 → Borrower had a delinquency (mths_since_last_delinq > 0)

0 → No delinquency (mths_since_last_delinq = 0)

Useful as a feature for credit risk modeling.

In [0]:
from pyspark.sql.functions import col, round

# Ensure the column exists
if "late_payment_flag" in df.columns:
    total_count = df.count()
    if total_count == 0:
        print("DataFrame is empty, cannot calculate distribution.")
    else:
        # Count borrowers per late payment flag
        late_payment_distribution = (
            df.groupBy("late_payment_flag")
              .count()
              .withColumn("percentage", round(col("count") / total_count * 100, 2))
        )
        
        # Show distribution
        late_payment_distribution.show()
else:
    print("Column 'late_payment_flag' not found in DataFrame")


+-----------------+------+----------+
|late_payment_flag| count|percentage|
+-----------------+------+----------+
|                0|441119|     51.53|
|                1|414850|     48.47|
+-----------------+------+----------+



In [0]:
# Categorize borrowers with late payments based on how many months ago their last delinquency occurred (mths_since_last_delinq)

from pyspark.sql.functions import when, col

# Ensure column exists
if "mths_since_last_delinq" in df.columns:
    # Replace nulls with 0
    df = df.fillna({"mths_since_last_delinq": 0})
    
    # Create categorical column
    df = df.withColumn(
        "late_payment_category",
        when(col("mths_since_last_delinq") == 0, "No Delinquency")
        .when((col("mths_since_last_delinq") >= 1) & (col("mths_since_last_delinq") <= 6), "Recent")
        .when((col("mths_since_last_delinq") >= 7) & (col("mths_since_last_delinq") <= 24), "Moderate")
        .otherwise("Old")
    )
    
    # Show sample
    df.select("mths_since_last_delinq", "late_payment_category").show(10)
else:
    print("Column 'mths_since_last_delinq' not found in DataFrame")


+----------------------+---------------------+
|mths_since_last_delinq|late_payment_category|
+----------------------+---------------------+
|                     0|       No Delinquency|
|                     0|       No Delinquency|
|                     0|       No Delinquency|
|                     0|       No Delinquency|
|                     0|       No Delinquency|
|                     0|       No Delinquency|
|                    31|                  Old|
|                     0|       No Delinquency|
|                    69|                  Old|
|                     0|       No Delinquency|
+----------------------+---------------------+
only showing top 10 rows


In [0]:
from pyspark.sql.functions import col, round

# Ensure the column exists
if "late_payment_category" in df.columns:
    total_count = df.count()
    if total_count == 0:
        print("DataFrame is empty, cannot calculate distribution.")
    else:
        # Count borrowers per category
        late_payment_dist = (
            df.groupBy("late_payment_category")
              .count()
              .withColumn("percentage", round(col("count") / total_count * 100, 2))
        )
        
        # Show distribution
        late_payment_dist.show()
else:
    print("Column 'late_payment_category' not found in DataFrame")


+---------------------+------+----------+
|late_payment_category| count|percentage|
+---------------------+------+----------+
|       No Delinquency|441119|     51.53|
|                  Old|249305|     29.13|
|             Moderate|133575|     15.61|
|               Recent| 31970|      3.73|
+---------------------+------+----------+



In [0]:
# Encoding addr_state into regions or create a risk index based on historical default rates by state.

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# -------------------------------
# Define regions
# -------------------------------
regions = {
    'Northeast': ['ME','NH','VT','MA','RI','CT','NY','NJ','PA'],
    'Midwest': ['OH','MI','IN','IL','WI','MN','IA','MO','ND','SD','NE','KS'],
    'South': ['DE','MD','DC','VA','WV','NC','SC','GA','FL','KY','TN','MS','AL','OK','TX','AR','LA'],
    'West': ['ID','MT','WY','NV','UT','CO','AZ','NM','WA','OR','CA','AK','HI']
}

def state_to_region(state):
    for region, states in regions.items():
        if state in states:
            return region
    return 'Other'

state_to_region_udf = F.udf(state_to_region, StringType())

df = df.withColumn("region", state_to_region_udf(F.col("addr_state")))

In [0]:
# -------------------------------
# Calculate state default rate
# -------------------------------
state_default = df.groupBy("addr_state").agg(F.mean("default_ind").alias("default_rate"))

df = df.join(state_default, on="addr_state", how="left")

In [0]:
# -------------------------------
# Create state risk bins
# -------------------------------
quantiles = df.approxQuantile("default_rate", [0.33, 0.66], 0.0)

df = df.withColumn(
    "state_risk",
    F.when(F.col("default_rate") <= quantiles[0], "Low")
     .when(F.col("default_rate") <= quantiles[1], "Medium")
     .otherwise("High")
)


In [0]:
# -------------------------------
# Combine region + risk
# -------------------------------
df = df.withColumn(
    "region_risk",
    F.concat_ws("_", F.col("region"), F.col("state_risk"))
)

In [0]:
print(df.columns)

['addr_state', 'id', 'member_id', 'issue_date', 'earliest_cr_line_parsed', 'lastpymt_date_parsed', 'lastcreditpull_date_parsed', 'next_payment_date_parsed', 'default_ind', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length_num', 'emp_title', 'home_ownership', 'annual_inc', 'verification_status', 'pymnt_plan', 'desc', 'purpose', 'title', 'zip_code', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_amnt', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m', 'o

In [0]:
# Installment-to-Income Ratio (monthly debt burden)
from pyspark.sql import functions as F

# Calculate monthly income
df = df.withColumn("monthly_income", F.col("annual_inc") / 12)

# Calculate Installment-to-Income ratio
df = df.withColumn("installment_to_income_ratio", F.col("installment") / F.col("monthly_income"))

# Optional: handle zero or null monthly_income to avoid division errors
df = df.withColumn(
    "installment_to_income_ratio",
    F.when(F.col("monthly_income") > 0, F.col("installment_to_income_ratio"))
     .otherwise(None)
)

df.select("installment_to_income_ratio").show(10)

# installment_to_income_ratio → fraction of monthly income used for loan installment

# Values > 1 indicate the borrower’s installment exceeds their monthly income (high burden).


+---------------------------+
|installment_to_income_ratio|
+---------------------------+
|       0.056908499999999994|
|        0.01842327272727273|
|        0.07815636363636363|
|        0.07204949999999999|
|        0.11501142857142857|
|         0.1406181818181818|
|                   0.028984|
|        0.08247897196261682|
|                 0.06138675|
|        0.06400454545454545|
+---------------------------+
only showing top 10 rows


In [0]:
from pyspark.sql import functions as F

# Define bins for monthly burden
# Example thresholds (you can adjust based on distribution)
# Low: <= 0.2, Medium: 0.2 - 0.4, High: > 0.4
df = df.withColumn(
    "monthly_burden_category",
    F.when(F.col("installment_to_income_ratio") <= 0.2, "Low")
     .when(F.col("installment_to_income_ratio") <= 0.4, "Medium")
     .when(F.col("installment_to_income_ratio") > 0.4, "High")
     .otherwise("Unknown")
)

df.select("monthly_burden_category").show(10)


+-----------------------+
|monthly_burden_category|
+-----------------------+
|                    Low|
|                    Low|
|                    Low|
|                    Low|
|                    Low|
|                    Low|
|                    Low|
|                    Low|
|                    Low|
|                    Low|
+-----------------------+
only showing top 10 rows


In [0]:
# Categorize int_rate into Low / Medium / High risk brackets and then summarize loans in each category.

from pyspark.sql import functions as F

# -------------------------------
# Convert interest rate to numeric if needed
# -------------------------------
# If int_rate is a string like '13.56%', remove '%' and convert to float
df = df.withColumn(
    "int_rate_numeric",
    F.regexp_replace(F.col("int_rate"), "%", "").cast("double")
)

# -------------------------------
# Define risk brackets
# -------------------------------
# Example thresholds (adjust based on your dataset)
# Low risk: <= 10%, Medium: 10-20%, High: > 20%
df = df.withColumn(
    "int_rate_risk",
    F.when(F.col("int_rate_numeric") <= 10, "Low")
     .when(F.col("int_rate_numeric") <= 20, "Medium")
     .otherwise("High")
)

# -------------------------------
# Summarize loans in each risk category
# -------------------------------
summary_df = df.groupBy("int_rate_risk").agg(
    F.count("*").alias("num_loans"),
    F.mean("loan_amnt").alias("avg_loan_amount"),
    F.mean("installment").alias("avg_installment"),
    F.mean("int_rate_numeric").alias("avg_int_rate")
)

# Show summary
summary_df.show()


+-------------+---------+------------------+------------------+------------------+
|int_rate_risk|num_loans|   avg_loan_amount|   avg_installment|      avg_int_rate|
+-------------+---------+------------------+------------------+------------------+
|       Medium|   570314|14630.160183688284| 431.0820867803184|14.391655929949893|
|          Low|   230582| 14064.39357798961|420.88172441905556|7.9709181982857285|
|         High|    55073|18792.707406533147| 553.9260089335872|22.633658961736746|
+-------------+---------+------------------+------------------+------------------+



In [0]:
# Credit activity rate: e.g., total_acc / credit_age_years → average new accounts per year.

from pyspark.sql import functions as F

# -------------------------------
# Ensure credit_age_years is not zero to avoid division errors
# -------------------------------
df = df.withColumn(
    "credit_activity_rate",
    F.when(F.col("credit_age_years") > 0, F.col("total_acc") / F.col("credit_age_years"))
     .otherwise(None)
)

df.select("credit_activity_rate").show(10)

+--------------------+
|credit_activity_rate|
+--------------------+
|   1.254220935841775|
|  0.8194138039710054|
|   1.721556886227545|
|  0.4635482511588706|
|  0.5827505827505828|
|  0.2799552071668533|
|   0.978067575577949|
| 0.48051254671649757|
|  0.8411843876177658|
|  0.8113590263691685|
+--------------------+
only showing top 10 rows


In [0]:
from pyspark.sql import functions as F

# -------------------------------
# Define bins for credit activity
# -------------------------------
# Example thresholds (adjust based on your dataset)
# Low: <= 1 account/year, Medium: 1-3 accounts/year, High: >3 accounts/year
df = df.withColumn(
    "credit_activity_category",
    F.when(F.col("credit_activity_rate") <= 1, "Low")
     .when(F.col("credit_activity_rate") <= 3, "Medium")
     .when(F.col("credit_activity_rate") > 3, "High")
     .otherwise("Unknown")
)

df.select("credit_activity_category").show(10)


+------------------------+
|credit_activity_category|
+------------------------+
|                  Medium|
|                     Low|
|                  Medium|
|                     Low|
|                     Low|
|                     Low|
|                     Low|
|                     Low|
|                     Low|
|                     Low|
+------------------------+
only showing top 10 rows


In [0]:
from pyspark.sql import functions as F

# -------------------------------
# Calculate recovery efficiency
# -------------------------------
df = df.withColumn(
    "recovery_efficiency",
    F.when(F.col("total_pymnt") > 0, F.col("recoveries") / F.col("total_pymnt"))
     .otherwise(None)
)

df.select("recovery_efficiency").show(10)


+-------------------+
|recovery_efficiency|
+-------------------+
|                0.0|
|                0.0|
|                0.0|
|                0.0|
|0.08585832316922525|
|                0.0|
|                0.0|
|                0.0|
|0.10810377770826225|
|                0.0|
+-------------------+
only showing top 10 rows


In [0]:
# Default risk interaction features

from pyspark.sql import functions as F

# -------------------------------
# Ensure default indicator is numeric
# -------------------------------
# Assume default_ind is 1 for default, 0 for no default
# If it's a string, convert it first
df = df.withColumn("default_ind_numeric", F.col("default_ind").cast("double"))

# -------------------------------
# Create interaction features
# -------------------------------
df = df.withColumn(
    "default_x_int_rate",
    F.col("default_ind_numeric") * F.col("int_rate_numeric")
)

df = df.withColumn(
    "default_x_revol_utilization",
    F.col("default_ind_numeric") * F.col("revol_utilization")
)


df.select("default_x_int_rate", "default_x_revol_utilization").show(10)

+------------------+---------------------------+
|default_x_int_rate|default_x_revol_utilization|
+------------------+---------------------------+
|               0.0|                        0.0|
|               0.0|                        0.0|
|               0.0|                        0.0|
|               0.0|                        0.0|
|             12.69|                        0.0|
|               0.0|                        0.0|
|               0.0|                        0.0|
|               0.0|                        0.0|
|             19.42|                        0.0|
|               0.0|                        0.0|
+------------------+---------------------------+
only showing top 10 rows


In [0]:
# Loan age: current_date - issue_date → how long the loan has been active.

from pyspark.sql import functions as F

# -------------------------------
# Ensure issue_date is in date format
# -------------------------------
# df = df.withColumn("issue_date", F.to_date(F.col("issue_date"), "yyyy-MM-dd"))

# -------------------------------
# Calculate loan age in days
# -------------------------------
df = df.withColumn(
    "loan_age_days",
    F.datediff(F.current_date(), F.col("issue_date"))
)

# -------------------------------
# Optional: Calculate loan age in years
# -------------------------------
df = df.withColumn(
    "loan_age_years",
    F.round(F.col("loan_age_days") / 365.25, 2)
)

df.select("loan_age_days", "loan_age_years").show(10)


+-------------+--------------+
|loan_age_days|loan_age_years|
+-------------+--------------+
|         3914|         10.72|
|         5375|         14.72|
|         5375|         14.72|
|         5375|         14.72|
|         5375|         14.72|
|         5375|         14.72|
|         5375|         14.72|
|         5375|         14.72|
|         5375|         14.72|
|         5375|         14.72|
+-------------+--------------+
only showing top 10 rows


In [0]:
df.printSchema()

root
 |-- addr_state: string (nullable = true)
 |-- id: long (nullable = true)
 |-- member_id: long (nullable = true)
 |-- issue_date: date (nullable = true)
 |-- earliest_cr_line_parsed: date (nullable = true)
 |-- lastpymt_date_parsed: date (nullable = true)
 |-- lastcreditpull_date_parsed: date (nullable = true)
 |-- next_payment_date_parsed: date (nullable = true)
 |-- default_ind: long (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- funded_amnt: double (nullable = true)
 |-- funded_amnt_inv: double (nullable = true)
 |-- term: integer (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_length_num: integer (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: double (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- pymnt_plan: string (nullabl

In [0]:
# Debt-to-Income + Loan-to-Income interaction → overall financial stress indicator.

from pyspark.sql import functions as F

# -------------------------------
# Ensure numeric columns
# -------------------------------
# Assume 'dti' = debt-to-income ratio (as fraction or %)
# 'loan_amnt' = loan amount
# 'annual_inc' = annual income

df = df.withColumn("annual_inc_numeric", F.col("annual_inc").cast("double"))
df = df.withColumn("loan_amnt_numeric", F.col("loan_amnt").cast("double"))
df = df.withColumn("dti_numeric", F.col("debt_to_income_ratio").cast("double"))

# -------------------------------
# Calculate Loan-to-Income ratio (LTI)
# -------------------------------
df = df.withColumn(
    "loan_to_income",
    F.when(F.col("annual_inc_numeric") > 0, F.col("loan_amnt_numeric") / F.col("annual_inc_numeric"))
     .otherwise(None)
)

# -------------------------------
# Create financial stress interaction: DTI * LTI
# -------------------------------
df = df.withColumn(
    "financial_stress_indicator",
    F.col("dti_numeric") * F.col("loan_to_income")
)



In [0]:
# Multiple loan features combined: e.g., loan_amnt / funded_amnt → underfunding ratio.
from pyspark.sql import functions as F

# -------------------------------
# 1️⃣ Ensure numeric columns
# -------------------------------
df = df.withColumn("loan_amnt_numeric", F.col("loan_amnt").cast("double"))
df = df.withColumn("funded_amnt_numeric", F.col("funded_amnt").cast("double"))

# -------------------------------
# 2️⃣ Calculate underfunding ratio
# -------------------------------
df = df.withColumn(
    "underfunding_ratio",
    F.when(F.col("funded_amnt_numeric") > 0, F.col("loan_amnt_numeric") / F.col("funded_amnt_numeric"))
     .otherwise(None)
)


In [0]:
from pyspark.sql import functions as F

# -------------------------------
# Ensure underfunding_ratio exists
# -------------------------------
# Already calculated as: loan_amnt / funded_amnt
# df = df.withColumn("underfunding_ratio", ...)

# -------------------------------
# Define bins for underfunding
# -------------------------------
df = df.withColumn(
    "underfunding_category",
    F.when(F.col("underfunding_ratio") == 1, "Fully Funded")
     .when((F.col("underfunding_ratio") < 1) & (F.col("underfunding_ratio") >= 0.9), "Slightly Underfunded")
     .when(F.col("underfunding_ratio") < 0.9, "Underfunded")
     .otherwise("Unknown")
)

# Fully Funded → ratio = 1
# Slightly Underfunded → ratio between 0.9 and <1
# Underfunded → ratio < 0.9
# Unknown → missing or invalid values

df.select("underfunding_ratio", "underfunding_category").show(10)

+------------------+---------------------+
|underfunding_ratio|underfunding_category|
+------------------+---------------------+
|               1.0|         Fully Funded|
|               1.0|         Fully Funded|
|               1.0|         Fully Funded|
|               1.0|         Fully Funded|
|               1.0|         Fully Funded|
|               1.0|         Fully Funded|
|               1.0|         Fully Funded|
|               1.0|         Fully Funded|
|               1.0|         Fully Funded|
|               1.0|         Fully Funded|
+------------------+---------------------+
only showing top 10 rows


In [0]:
# Save Transformed table to Catalog
df.write.saveAsTable(f"data_processed.credit_risk_main.Loans_FeatureEngineered")